# A/B Test: Model Targeted vs Uniform Outreach

**Business question:** Given a fixed budget to email 300 free members with an Academy
nudge, does emailing the model's top 300 picks convert more than emailing 300 random users?

**Design:**
- A fresh cohort of 1,000 free members is generated (a forward looking population the
  model has never seen).
- All 1,000 are scored with the propensity model from the previous stage.
- Uniform arm (control): 300 users chosen at random receive the nudge.
- Targeted arm (treatment): the top 300 users by model score receive the same nudge.
- Same message, same cost. The only difference is who was selected.
- A two proportion z test determines whether the conversion difference is statistically significant.

**Data required:** `conversion_dataset.csv` (to retrain the model for scoring the new cohort).


## 1. Upload the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

import io, pandas as pd, numpy as np
fname = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[fname]))
print(f"Loaded '{fname}'  ->  {df.shape[0]:,} rows x {df.shape[1]} columns")


## 2. Retrain the model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

target = 'upgraded'
drop_cols = ['user_id', 'university_name', 'major_name', target]
categorical = ['profile_source', 'school_tier', 'class_year', 'major_cat']
numeric = [c for c in df.columns if c not in drop_cols + categorical]

X = df[categorical + numeric]
y = df[target]

preprocess = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop=None), categorical),
])

model = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000,
                               random_state=RANDOM_STATE)),
])
model.fit(X, y)
print("Model retrained on full dataset.")


## 3. Generate a fresh cohort

1,000 new free members the model has never seen. Each user has behavioral features
(visible to the model) and a hidden baseline conversion probability plus a
responsiveness value (used only to simulate realistic outcomes).


In [ ]:
RNG = np.random.default_rng(99999)
N_NEW = 1000

intent   = np.clip(RNG.normal(0, 1, N_NEW), -3, 3)
activity = np.clip(0.35 * intent + RNG.normal(0, 1, N_NEW), -3, 3)
engagement = 0.6 * intent + 0.4 * activity

p_has_profile = 1 / (1 + np.exp(-(0.9 * engagement + 0.75)))
has_profile = RNG.random(N_NEW) < p_has_profile

KNOWN_SOURCES = ["resume_template_form", "info_session_registration",
                 "resume_workshop_registration", "email_capture", "self_completed_profile"]

def pick_source(eng):
    base = np.array([0.34, 0.18, 0.16, 0.20, 0.12])
    tilt = np.array([0.00, 0.10, 0.12, -0.06, 0.02]) * eng
    w = np.clip(base + tilt, 0.01, None); w = w / w.sum()
    return RNG.choice(KNOWN_SOURCES, p=w)

profile_source = np.where(has_profile, [pick_source(e) for e in engagement], "unknown")

YEARS = ["freshman", "sophomore", "junior", "senior"]
school_tier, class_year, major_cat = [], [], []
for i in range(N_NEW):
    if not has_profile[i]:
        school_tier.append("unknown"); class_year.append("unknown"); major_cat.append("unknown")
        continue
    p_t = 1/(1+np.exp(-(0.8*intent[i]-0.2)))
    r = RNG.random()
    school_tier.append("target" if r < p_t*0.6 else ("semi_target" if r < p_t*0.6+0.4 else "non_target"))
    p_f = 1/(1+np.exp(-(0.7*intent[i])))
    r2 = RNG.random()
    major_cat.append("finance_econ" if r2 < p_f*0.6 else ("stem" if r2 < p_f*0.6+0.30 else "other"))
    w = np.array([0.15,0.22,0.38,0.25]) + np.array([-0.05,-0.02,0.05,0.02])*intent[i]
    w = np.clip(w, 0.02, None); w = w/w.sum()
    class_year.append(RNG.choice(YEARS, p=w))

def poisson_from(latent, base, scale, floor=0):
    lam = np.clip(base + scale * latent, 0.05, None)
    return np.maximum(floor, RNG.poisson(lam))

mentor_booked      = (RNG.random(N_NEW) < 1/(1+np.exp(-(1.1*intent-1.4)))).astype(int)
live_resume_used   = (RNG.random(N_NEW) < 1/(1+np.exp(-(1.0*intent-1.8)))).astype(int)
resume_tool_used   = (RNG.random(N_NEW) < 1/(1+np.exp(-(0.9*intent-0.6)))).astype(int)
course_previews    = poisson_from(intent, 1.5, 1.8)
target_co_searches = poisson_from(intent, 2.0, 2.6)
logins             = poisson_from(activity, 8.0, 3.0, floor=1)
content_views      = poisson_from(activity, 12.0, 6.0, floor=1)
forum_posts        = poisson_from(activity, 1.0, 1.2)
tenure_days        = np.clip(RNG.normal(120+20*activity, 60), 5, 400).astype(int)
days_since_active  = np.clip(RNG.normal(20-8*intent, 12), 0, 120).astype(int)
email_open_rate    = np.round(np.clip(1/(1+np.exp(-(0.8*engagement)))+RNG.normal(0,0.08,N_NEW),0.01,0.99),3)
email_click_rate   = np.round(np.clip(email_open_rate*(0.35+0.25*RNG.random(N_NEW)),0.0,0.98),3)

intent_events = course_previews + target_co_searches + resume_tool_used + mentor_booked + live_resume_used
total_events  = intent_events + logins + content_views + forum_posts
intent_ratio  = np.where(total_events > 0, np.round(intent_events / total_events, 3), 0.0)

SOURCE_LOGIT = {"resume_workshop_registration":0.45, "info_session_registration":0.40,
                "self_completed_profile":0.25, "resume_template_form":0.15,
                "email_capture":0.10, "unknown":-0.20}
TIER_LOGIT   = {"target":0.45, "semi_target":0.20, "non_target":0.0, "unknown":0.0}
YEAR_LOGIT   = {"junior":0.35, "senior":0.20, "sophomore":0.05, "freshman":0.0, "unknown":0.0}
MAJOR_LOGIT  = {"finance_econ":0.30, "stem":0.10, "other":0.0, "unknown":0.0}

logit = (
    -4.75
    + 0.95*mentor_booked + 0.75*live_resume_used + 0.60*resume_tool_used
    + 0.16*course_previews + 0.12*target_co_searches
    + np.array([TIER_LOGIT[t] for t in school_tier])
    + np.array([YEAR_LOGIT[y] for y in class_year])
    + np.array([MAJOR_LOGIT[m] for m in major_cat])
    + np.array([SOURCE_LOGIT[s] for s in profile_source])
    + 0.70*email_open_rate
    + 0.010*logins + 0.0015*np.round(np.clip(RNG.normal(70+40*activity,35),3,None),1)
    + 0.004*content_views - 0.010*days_since_active
    + RNG.normal(0, 0.35, N_NEW)
)
latent_p = 1 / (1 + np.exp(-logit))
responsiveness = np.round(1/(1+np.exp(-(0.9*intent - 0.3))), 4)

new_users = pd.DataFrame({
    'profile_source': profile_source, 'school_tier': school_tier,
    'class_year': class_year, 'major_cat': major_cat,
    'has_profile': has_profile.astype(int),
    'mentor_booked': mentor_booked, 'live_resume_used': live_resume_used,
    'resume_tool_used': resume_tool_used, 'course_previews': course_previews,
    'target_co_searches': target_co_searches, 'logins': logins,
    'content_views': content_views, 'forum_posts': forum_posts,
    'total_events': total_events, 'intent_events': intent_events,
    'intent_ratio': intent_ratio, 'tenure_days': tenure_days,
    'days_since_active': days_since_active, 'email_open_rate': email_open_rate,
    'email_click_rate': email_click_rate, 'has_email_record': 1,
})

print(f"Fresh cohort generated: {N_NEW} new free members")
print(f"Baseline conversion rate (no outreach): {latent_p.mean()*100:.1f}%")


## 4. Score the fresh cohort

In [ ]:
X_new = new_users[categorical + numeric]
propensity_scores = model.predict_proba(X_new)[:, 1]

new_users['propensity'] = propensity_scores
new_users['latent_p'] = latent_p
new_users['responsiveness'] = responsiveness

print(f"Scored {N_NEW} users.")
print(f"Propensity score range: {propensity_scores.min():.3f} to {propensity_scores.max():.3f}")


## 5. Run the experiment

Budget: 300 emails. The outreach effect is modulated by each user's responsiveness,
reflecting the assumption that high intent users are also more movable by the nudge.


In [ ]:
BUDGET = 300
OUTREACH_EFFECT = 0.10

uniform_idx = RNG.choice(N_NEW, size=BUDGET, replace=False)
targeted_idx = np.argsort(propensity_scores)[::-1][:BUDGET]

def simulate_arm(indices, label):
    base_p = latent_p[indices]
    resp   = responsiveness[indices]
    boosted_p = np.clip(base_p + OUTREACH_EFFECT * resp, 0, 1)
    outcomes = (RNG.random(len(indices)) < boosted_p).astype(int)
    conv_rate = outcomes.mean()
    n_converted = outcomes.sum()
    print(f"{label}:")
    print(f"  Users emailed:    {len(indices)}")
    print(f"  Conversions:      {n_converted}")
    print(f"  Conversion rate:  {conv_rate*100:.1f}%")
    print()
    return outcomes, conv_rate

print("=" * 50)
print("A/B TEST RESULTS")
print("=" * 50)
print()
uniform_outcomes, uniform_rate   = simulate_arm(uniform_idx, "UNIFORM ARM (control)")
targeted_outcomes, targeted_rate = simulate_arm(targeted_idx, "TARGETED ARM (treatment)")

abs_lift = targeted_rate - uniform_rate
rel_lift = abs_lift / uniform_rate * 100 if uniform_rate > 0 else float('inf')

print(f"Absolute lift:  {abs_lift*100:+.1f} percentage points")
print(f"Relative lift:  {rel_lift:+.1f}%")


## 6. Statistical significance test

Two proportion z test. Null hypothesis: there is no difference between targeting
strategies. If p value < 0.05, the lift is statistically significant.


In [ ]:
from scipy import stats

n1 = len(targeted_outcomes)
n2 = len(uniform_outcomes)
p1 = targeted_rate
p2 = uniform_rate

p_pool = (targeted_outcomes.sum() + uniform_outcomes.sum()) / (n1 + n2)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
z = (p1 - p2) / se
p_value = 1 - stats.norm.cdf(z)

se_diff = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
ci_low  = (p1 - p2) - 1.96 * se_diff
ci_high = (p1 - p2) + 1.96 * se_diff

print("TWO PROPORTION Z TEST")
print("=" * 50)
print(f"Targeted conversion rate:  {p1*100:.1f}%")
print(f"Uniform conversion rate:   {p2*100:.1f}%")
print(f"Difference:                {(p1-p2)*100:+.1f} pp")
print(f"z statistic:               {z:.3f}")
print(f"p value (one sided):       {p_value:.4f}")
print(f"95% confidence interval:   [{ci_low*100:+.1f}, {ci_high*100:+.1f}] pp")
print()
if p_value < 0.05:
    print(f"RESULT: Statistically significant (p = {p_value:.4f} < 0.05).")
    print(f"Model targeted outreach converts significantly better than uniform.")
else:
    print(f"RESULT: Not statistically significant (p = {p_value:.4f}).")


## 7. Visualize

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
bars = ax.bar(['Uniform\n(random 300)', 'Targeted\n(top 300 by model)'],
              [uniform_rate*100, targeted_rate*100],
              color=['#e76f51', '#1B3A5C'], width=0.5)
ax.set_ylabel('Conversion rate (%)')
ax.set_title('A/B Test: Conversion Rates')
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
            f'{b.get_height():.1f}%', ha='center', fontweight='bold')
mid_y = (uniform_rate*100 + targeted_rate*100) / 2
ax.annotate(f'+{abs_lift*100:.0f} pp lift', xy=(1, targeted_rate*100),
            xytext=(1.35, mid_y), fontsize=12, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#333'),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#f0f0f0'))

ax2 = axes[1]
ax2.hist(propensity_scores[uniform_idx], bins=20, alpha=0.6, color='#e76f51',
         label='Uniform (random)', density=True)
ax2.hist(propensity_scores[targeted_idx], bins=20, alpha=0.6, color='#1B3A5C',
         label='Targeted (top scores)', density=True)
ax2.set_xlabel('Model propensity score')
ax2.set_ylabel('Density')
ax2.set_title('Who was selected in each arm')
ax2.legend()

plt.tight_layout()
plt.show()


## 8. Save results

In [ ]:
results = {
    'uniform_conversion_rate': round(uniform_rate, 4),
    'targeted_conversion_rate': round(targeted_rate, 4),
    'absolute_lift_pp': round(abs_lift * 100, 2),
    'relative_lift_pct': round(rel_lift, 2),
    'z_statistic': round(z, 4),
    'p_value': round(p_value, 4),
    'ci_95_low_pp': round(ci_low * 100, 2),
    'ci_95_high_pp': round(ci_high * 100, 2),
    'budget': BUDGET,
    'cohort_size': N_NEW,
    'significant': bool(p_value < 0.05),
}

import json
with open('ab_test_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Saved: ab_test_results.json")
print()
print(json.dumps(results, indent=2))

from google.colab import files as _f
_f.download('ab_test_results.json')
